# 🌊 Americas TechGuard - MVP Simulador Híbrido LoRaWAN / Meshtastic (Atividade 8)
**Autor:** Nyrx Oliveira de Aquino Farias  
**Contexto:** Este notebook demonstra a simulação de uma arquitetura de sensoriamento e rede resiliente e híbrida. Em cenários normais, os dados são transmitidos via arquitetura inspirada em LoRaWAN. Em cenários de desastre off-grid (queda do gateway), o sistema chaveia automaticamente para a rede mesh Meshtastic.

Nesta simulação, cobriremos:
1. Coleta síncrona de 4 variáveis ambientais: Nível da água, Chuva, Temperatura e Umidade.
2. Classificação em tempo real do nível de risco (SAFE, ATTENTION, ALERT, CRITICAL).
3. Adaptação do payload JSON e simulação de envio para tópicos MQTT específicos de cada rede.

### Importações e Configurações Globais do Dispositivo

In [1]:
# Importação de bibliotecas nativas para simulação de tempo, aleatoriedade e formatação de dados
import json
import time
import random
from datetime import datetime

# Configurações de geolocalização fixa do nó sensor (Florianópolis)
DEVICE_ID = "NODE_FLORIPA_001"
LATITUDE = -27.5954
LONGITUDE = -48.5480

print(f"Configurações carregadas para o dispositivo: {DEVICE_ID}")

Configurações carregadas para o dispositivo: NODE_FLORIPA_001


### Inteligência de Borda - Regra de Classificação de Risco

In [2]:
def calcular_risco(water_level):
    """
    Algoritmo de tomada de decisão local (Edge Computing).
    Classifica o estado com base na leitura do sensor ultrassônico.
    """
    if water_level < 1.0:
        return "SAFE", "Nivel da agua normal. Monitoramento estavel."
    elif 1.0 <= water_level < 2.0:
        return "ATTENTION", "Atencao: Nivel do rio subindo de forma constante."
    elif 2.0 <= water_level < 3.0:
        return "ALERT", "ALERTA: Risco moderado de transbordo nas proximidades."
    else:
        return "CRITICAL", "ALERTA: Risco iminente de enchente! Transbordo detectado."

### Simulador Físico de Sensores e Comportamento de Rede

In [3]:
def simular_sensores_e_rede(passo):
    """
    Gera dados ambientais correlacionados e simula a degradação de sinal 
    e perda de infraestrutura conforme o tempo (passos de simulação).
    """
    # Passos 1 e 2: Condições Normais (Operação via LoRaWAN)
    if passo <= 2:
        water_level = round(random.uniform(0.3, 0.8), 2)
        rain = round(random.uniform(0.0, 5.0), 1)
        temperature = round(random.uniform(21.0, 24.0), 1)
        humidity = random.randint(70, 80)
        
        gateway_available = True
        network_mode = "LoRaWAN"
        mesh_hops = 0
        battery = random.randint(95, 100)
        rssi = random.randint(-75, -70)
        snr = round(random.uniform(8.0, 11.0), 1)
        
    # Passo 3: Transição (Chuva aumentando, sinal de rede oscilando)
    elif passo == 3:
        water_level = round(random.uniform(1.2, 1.8), 2)
        rain = round(random.uniform(15.0, 25.0), 1)
        temperature = round(random.uniform(20.0, 21.5), 1)
        humidity = random.randint(85, 90)
        
        gateway_available = True
        network_mode = "LoRaWAN"
        mesh_hops = 0
        battery = random.randint(90, 94)
        rssi = random.randint(-95, -90)  # Perda de qualidade do sinal
        snr = round(random.uniform(3.0, 6.0), 1)
        
    # Passos 4 em diante: Desastre Natural (Queda do Gateway e transição para Rede Mesh Meshtastic)
    else:
        water_level = round(random.uniform(3.0, 3.6), 2)
        rain = round(random.uniform(40.0, 55.0), 1)
        temperature = round(random.uniform(18.0, 19.5), 1)
        humidity = random.randint(92, 98)
        
        gateway_available = False  # Queda de energia e internet local
        network_mode = "Meshtastic"
        mesh_hops = random.randint(1, 3)  # Encaminhamento por saltos
        battery = random.randint(80, 89)
        rssi = random.randint(-118, -110) # Sinal fraco devido ao cenário crítico
        snr = round(random.uniform(-5.0, -2.0), 1)

    risk_level, alert_msg = calcular_risco(water_level)
    
    # Payload plano (flat) e estruturado no formato JSON MVP
    payload = {
        "device_id": DEVICE_ID,
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "latitude": LATITUDE,
        "longitude": LONGITUDE,
        "sensor_type": "ultrasonic_level",
        "sensor_value": water_level,
        "unit": "m",
        "water_level_m": water_level,
        "rain_mm": rain,
        "temperature_c": temperature,
        "humidity_pct": humidity,
        "risk_level": risk_level,
        "alert_message": alert_msg,
        "gateway_active": gateway_available,
        "gateway_available": gateway_available,
        "network_mode": network_mode,
        "mesh_hops": mesh_hops,
        "battery_percent": battery,
        "signal_rssi": rssi,
        "signal_snr": snr
    }
    
    return payload

### Roteador de Payload / Simulação de Protocolo MQTT

In [4]:
def simular_publicacao_mqtt(payload):
    """
    Simula a publicação de dados em brokers locais ou na nuvem (MQTT).
    Demonstra a inteligência de rede publicando em tópicos correspondentes ao meio de transmissão.
    """
    if payload["gateway_available"]:
        topico = f"americas_techguard/lorawan/{payload['device_id']}/telemetry"
    else:
        topico = f"americas_techguard/meshtastic/mesh_node/alerts"
        
    print(f"\n📡 [BROKER MQTT SIMULADO] Publicando no topico: {topico}")
    print(json.dumps(payload, indent=2, ensure_ascii=False))

### Execução Loop de Monitoramento

In [5]:
# Bloco de execução sequencial da simulação
print("=" * 75)
print("     INICIANDO EXECUÇÃO DO SISTEMA DE MONITORAMENTO AMERIDAS TECHGUARD     ")
print("=" * 75)

# Executa 6 ciclos de leitura para ilustrar perfeitamente a transição das redes
for passo in range(1, 7):
    print("-" * 75)
    print(f"🕒 Ciclo {passo}/6 | Processando dados no Edge Node...")
    
    # Gera o payload com base na simulação e na regra de risco
    payload = simular_sensores_e_rede(passo)
    
    # Interface amigável para depuração rápida
    status_icon = "🟢" if payload["risk_level"] == "SAFE" else "🟡" if payload["risk_level"] == "ATTENTION" else "🔴"
    print(f"Status do Risco: {status_icon} {payload['risk_level']}")
    print(f"Canal de Comunicação Ativo: {payload['network_mode']}")
    
    # Envia o payload simulado via MQTT
    simular_publicacao_mqtt(payload)
    
    # Intervalo reduzido para execução rápida dentro do Notebook
    time.sleep(2)

print("\n" + "=" * 75)
print("            SIMULAÇÃO CONCLUÍDA COM SUCESSO! (EVIDÊNCIA COMPLETA)           ")
print("=" * 75)

     INICIANDO EXECUÇÃO DO SISTEMA DE MONITORAMENTO AMERIDAS TECHGUARD     
---------------------------------------------------------------------------
🕒 Ciclo 1/6 | Processando dados no Edge Node...
Status do Risco: 🟢 SAFE
Canal de Comunicação Ativo: LoRaWAN

📡 [BROKER MQTT SIMULADO] Publicando no topico: americas_techguard/lorawan/NODE_FLORIPA_001/telemetry
{
  "device_id": "NODE_FLORIPA_001",
  "timestamp": "2026-07-15T21:16:26.207214Z",
  "latitude": -27.5954,
  "longitude": -48.548,
  "sensor_type": "ultrasonic_level",
  "sensor_value": 0.46,
  "unit": "m",
  "water_level_m": 0.46,
  "rain_mm": 3.6,
  "temperature_c": 23.1,
  "humidity_pct": 78,
  "risk_level": "SAFE",
  "alert_message": "Nivel da agua normal. Monitoramento estavel.",
  "gateway_active": true,
  "gateway_available": true,
  "network_mode": "LoRaWAN",
  "mesh_hops": 0,
  "battery_percent": 95,
  "signal_rssi": -72,
  "signal_snr": 8.4
}


C:\Users\nyrx_farias\AppData\Local\Temp\ipykernel_11048\3391202753.py:53: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "timestamp": datetime.utcnow().isoformat() + "Z",


---------------------------------------------------------------------------
🕒 Ciclo 2/6 | Processando dados no Edge Node...
Status do Risco: 🟢 SAFE
Canal de Comunicação Ativo: LoRaWAN

📡 [BROKER MQTT SIMULADO] Publicando no topico: americas_techguard/lorawan/NODE_FLORIPA_001/telemetry
{
  "device_id": "NODE_FLORIPA_001",
  "timestamp": "2026-07-15T21:16:28.213537Z",
  "latitude": -27.5954,
  "longitude": -48.548,
  "sensor_type": "ultrasonic_level",
  "sensor_value": 0.66,
  "unit": "m",
  "water_level_m": 0.66,
  "rain_mm": 3.9,
  "temperature_c": 21.6,
  "humidity_pct": 76,
  "risk_level": "SAFE",
  "alert_message": "Nivel da agua normal. Monitoramento estavel.",
  "gateway_active": true,
  "gateway_available": true,
  "network_mode": "LoRaWAN",
  "mesh_hops": 0,
  "battery_percent": 95,
  "signal_rssi": -72,
  "signal_snr": 10.3
}
---------------------------------------------------------------------------
🕒 Ciclo 3/6 | Processando dados no Edge Node...
Status do Risco: 🟡 ATTENTION
C